# Data Processing

This section focuses on the steps required to process the data before analysis. The key steps include:

1. **Data Cleaning**: Handle missing values, remove duplicates, and correct inconsistencies in the dataset.
2. **Data Transformation**: Normalize, scale, or encode data as needed for analysis or modeling.
3. **Feature Engineering**: Create new features or modify existing ones to improve model performance.

Ensure that all transformations are reproducible and well-documented for future reference.

In [ ]:
# Load the pandas library for data manipulation.
import pandas as pd


In [2]:
# Load the dataset
df_cost = pd.read_csv('Assets/Avg Cost by state/nces330_20.csv')
df_cost.head()

,Year,State,Type,Length,Expense,Value
0,2013,Alabama,Private,4-year,Fees/Tuition,13983
1,2013,Alabama,Private,4-year,Room/Board,8503
2,2013,Alabama,Public In-State,2-year,Fees/Tuition,4048
3,2013,Alabama,Public In-State,4-year,Fees/Tuition,8073
4,2013,Alabama,Public In-State,4-year,Room/Board,8473


### Encode Categorical Columns
Convert categorical variables (State, Type, Length, Expense) into dummy variables, dropping the first category to avoid multicollinearity.

In [ ]:
cat_cols = ['State', 'Type', 'Length', 'Expense']
df_clean = pd.get_dummies(df_cost, columns=cat_cols, prefix=['state', 'type', 'length', 'expense'], drop_first=True)
df_clean.head()  # Check the result

,Year,Value,state_Alaska,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_District of Columbia,...,state_Vermont,state_Virginia,state_Washington,state_West Virginia,state_Wisconsin,state_Wyoming,type_Public In-State,type_Public Out-of-State,length_4-year,expense_Room/Board
0,2013,13983,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,2013,8503,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
2,2013,4048,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,2013,8073,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,True,False
4,2013,8473,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,True,True


### Convert Booleans to Integers
Ensure dummy variables are integers (0/1) instead of booleans for consistency.

In [4]:
bool_cols = [col for col in df_clean.columns if col.startswith(('state_', 'type_', 'length_', 'expense_'))]
for col in bool_cols:
    df_clean[col] = df_clean[col].astype(int)

### Ensure Numeric Columns
Convert Year and Value to numeric types, handling any non-numeric entries gracefully.

In [ ]:
df_clean['Year'] = pd.to_numeric(df_clean['Year'], errors='coerce')
df_clean['Value'] = pd.to_numeric(df_clean['Value'], errors='coerce')
df_clean.dtypes  #Verify column types

Year                          int64
Value                         int64
state_Alaska                  int64
state_Arizona                 int64
state_Arkansas                int64
state_California              int64
state_Colorado                int64
state_Connecticut             int64
state_Delaware                int64
state_District of Columbia    int64
state_Florida                 int64
state_Georgia                 int64
state_Hawaii                  int64
state_Idaho                   int64
state_Illinois                int64
state_Indiana                 int64
state_Iowa                    int64
state_Kansas                  int64
state_Kentucky                int64
state_Louisiana               int64
state_Maine                   int64
state_Maryland                int64
state_Massachusetts           int64
state_Michigan                int64
state_Minnesota               int64
state_Mississippi             int64
state_Missouri                int64
state_Montana               

### Check Data Consistency
Identify negative values in 'Value' and years outside the 2000-2020 range.

In [6]:
negatives = df_clean[df_clean['Value'] < 0]
out_of_range = df_clean[(df_clean['Year'] < 2000) | (df_clean['Year'] > 2020)]
print(f"Negative Values: {negatives.shape[0]} rows")
print(f"Out-of-Range Years: {out_of_range.shape[0]} rows")

Negative Values: 0 rows
Out-of-Range Years: 345 rows


### Drop Invalid Rows
Remove rows with negative values or years outside 2000-2020.

In [7]:
df_clean = df_clean[(df_clean['Value'] >= 0) & 
                    (df_clean['Year'].between(2000, 2020))]
df_clean.shape  # Check new size

(3203, 56)

### Define Outlier Removal Function
Create a function to remove outliers from a column using the IQR method.

In [8]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"{column} - Lower: {lower}, Upper: {upper}")
    print(f"Rows outside:", df[(df[column] < lower) | (df[column] > upper)].shape[0])
    return df[(df[column] >= lower) & (df[column] <= upper)]

### Remove Outliers from Value
Apply the outlier removal function to the 'Value' column.

In [9]:
df_clean = remove_outliers(df_clean, 'Value')

Value - Lower: -2576.75, Upper: 24801.25
Rows outside: 388


### Apply Domain-Specific Cap
Restrict 'Value' to a reasonable range (0 to 75,000) based on domain knowledge.

In [ ]:
df_clean = df_clean[(df_clean['Value'] >= 0) & (df_clean['Value'] <= 75000)]
df_clean.describe()  # Summary stats

,Year,Value,state_Alaska,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_District of Columbia,...,state_Vermont,state_Virginia,state_Washington,state_West Virginia,state_Wisconsin,state_Wyoming,type_Public In-State,type_Public Out-of-State,length_4-year,expense_Room/Board
count,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,...,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000
mean,2016.417407,10227.934636,0.020249,0.021314,0.022735,0.017052,0.019893,0.017052,0.017052,0.008526,...,0.017052,0.019893,0.017052,0.022735,0.019183,0.018828,0.425933,0.365542,0.722202,0.426288
std,2.282512,4959.059654,0.140875,0.144456,0.149085,0.129486,0.139659,0.129486,0.129486,0.091957,...,0.129486,0.139659,0.129486,0.149085,0.137192,0.135940,0.494571,0.481667,0.447993,0.494625
min,2013.000000,1225.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2014.000000,7284.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2016.000000,9506.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,2018.000000,12125.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
max,2020.000000,24791.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Remove Duplicates
Drop any duplicate rows to ensure data integrity.

In [ ]:
df_clean = df_clean.drop_duplicates()
df_clean.shape  # Final size check

(2815, 56)

### Save the Cleaned Dataset
Export the processed data to a new CSV file for later use (e.g., model training).

In [12]:
df_clean.to_csv('Assets/Avg Cost by state/cleaned.csv', index=False)
print("Data saved successfully!")

Data saved successfully!
